# Model Evaluation with K-Fold & Grid Search

- Cancer Dataset (in-built dataset)

### Step 1: Import necessary libraries

In [43]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

### Step 2: Load the Cancer dataset

In [44]:
"""
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)"""

'\nfrom sklearn.datasets import load_breast_cancer\ndata = load_breast_cancer()\nX = pd.DataFrame(data.data, columns=data.feature_names)\ny = pd.Series(data.target)'

In [45]:
# Load dataset from CSV
data = pd.read_csv("cancer.csv")


In [46]:
# Features (exclude both target/diagnosis columns)
X = data.drop(columns=['target', 'diagnosis'])

# Target: use diagnosis column and encode it
y = data['diagnosis']  # contains 'benign'/'malignant'
le = LabelEncoder()
y = le.fit_transform(y)  # 'benign' -> 0, 'malignant' -> 1

In [47]:
# Encode target if it is string
if y.dtype == 'object':
    le = LabelEncoder()
    y = le.fit_transform(y)  # 'benign'->0, 'malignant'->1

### Step 3: Split the dataset into training and testing sets

In [48]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### Step 4: Create a Logistic Regression classifier

In [49]:
clf = LogisticRegression(random_state=42)

### Step 5: Perform K-Fold Cross-Validation (K=5) to evaluate the model


In [50]:
cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')

### Step 6: Fit the model on the entire training set


In [51]:
clf.fit(X_train, y_train)


LogisticRegression(random_state=42)

### Step 7: Make predictions on the test set

In [52]:
y_pred = clf.predict(X_test)


### Step 8: Evaluate the model using classification metrics


In [53]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

### Step 9: Print the K-Fold Cross-Validation results

In [54]:
print("K-Fold Cross-Validation Results:")
for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score:.2f}")

K-Fold Cross-Validation Results:
Fold 1: 0.97
Fold 2: 0.90
Fold 3: 0.97
Fold 4: 0.95
Fold 5: 0.90


### Step 10: Print the evaluation metrics and classification report

In [55]:
print("\nTest Set Evaluation Metrics:")
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Test Set Evaluation Metrics:
Accuracy: 0.96
Precision: 0.98
Recall: 0.93
F1 Score: 0.95

Confusion Matrix:
[[70  1]
 [ 3 40]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        71
           1       0.98      0.93      0.95        43

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



### Step 11: Perform Grid Search for hyperparameter tuning

In [56]:
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(random_state=42),
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100]},
             scoring='accuracy')

### Step 12: Get the best hyperparameters

In [57]:
best_params = grid_search.best_params_

### Step 13: Train a Logistic Regression classifier with the best hyperparameters

In [58]:
best_clf = LogisticRegression(random_state=42, **best_params)
best_clf.fit(X_train, y_train)

LogisticRegression(C=100, random_state=42)

### Step 14: Make predictions with the tuned model

In [59]:
y_pred_tuned = best_clf.predict(X_test)


### Step 15: Evaluate the tuned model

In [60]:
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
print("\nGrid Search Results:")
print(f"Best Hyperparameters: {best_params}")
print(f"Test Set Accuracy with Tuned Model: {accuracy_tuned:.2f}")


Grid Search Results:
Best Hyperparameters: {'C': 100}
Test Set Accuracy with Tuned Model: 0.96


### Compare result of k-fold & Grid Search

In [61]:
# Perform K-Fold Cross-Validation and store the accuracy scores
cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')

# Calculate the accuracy of the tuned model
accuracy_tuned = accuracy_score(y_test, y_pred_tuned)

# Print and compare the results side by side
print("K-Fold Cross-Validation Results:")
print(f"Mean Accuracy: {np.mean(cv_scores):.2f}")
print(f"Test Set Accuracy: {accuracy:.2f}")

print("\nGrid Search Results:")
print(f"Best Hyperparameters: {best_params}")
print(f"Test Set Accuracy with Tuned Model: {accuracy_tuned:.2f}")

K-Fold Cross-Validation Results:
Mean Accuracy: 0.94
Test Set Accuracy: 0.96

Grid Search Results:
Best Hyperparameters: {'C': 100}
Test Set Accuracy with Tuned Model: 0.96
